In [1]:
"""
The Goal of this notebook:
- add a second camera to the satellite
-- different mount (location, orientation) /FOV
-- 

Verification:
- show static rendering of satellite (visibility cone)
--verify mount is correct (visual, forward looking)
--verify changing angle works without fucking up sim
- show classification view array in simulation (video)
-- test take

API:
- go through SimConfig (can be consumed by Sim Kernel)
-- define what is needed (keep running in sim kernel in mind, should tie in SimConfig, can be extended, can be passed as function)
-- (instead of passing target area, where target area is currently parsed send this function)


use defaults:
different color than main camera
(I think we have defined the default already in the first notebook for FOV
and mount point of the camera -> Look up)

sampling_time: 200 microseconds
-> simplification, the time of the image taken is instant,
use closest timestep after cmd release to get current quality

"""

'\nThe Goal of this notebook:\n- add a second camera to the satellite\n-- different mount (location, orientation) /FOV\n-- \n\nVerification:\n- show static rendering of satellite (visibility cone)\n--verify mount is correct (visual, forward looking)\n--verify changing angle works without fucking up sim\n- show classification view array in simulation (video)\n-- test take\n\nAPI:\n- go through SimConfig (can be consumed by Sim Kernel)\n-- define what is needed (keep running in sim kernel in mind, should tie in SimConfig, can be extended, can be passed as function)\n-- (instead of passing target area, where target area is currently parsed send this function)\n\n\nuse defaults:\ndifferent color than main camera\n(I think we have defined the default already in the first notebook for FOV\nand mount point of the camera -> Look up)\n\nsampling_time: 200 microseconds\n-> simplification, the time of the image taken is instant,\nuse closest timestep after cmd release to get current quality\n\n'

In [2]:
import os
import sys
from pathlib import Path

notebook_dir = Path.cwd()
backend_root = notebook_dir
for _ in range(6):
    if (backend_root / "simulation").is_dir():
        break
    backend_root = backend_root.parent
os.chdir(backend_root)
sys.path.insert(0, str(backend_root))
_s01_dir = backend_root / "notebooks" / "s01"
sys.path.insert(0, str(_s01_dir))
ARTIFACT_DIR = _s01_dir / "artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
print(f"backend_root={backend_root}")
print(f"artifact_dir={ARTIFACT_DIR}")

backend_root=d:\code\sem-proj-asc\backend
artifact_dir=d:\code\sem-proj-asc\backend\notebooks\s01\artifacts


In [3]:
# s01 secondary camera preset (shared with build_setup).
from environment_definition.constants.SATELLITE import CAMERA_ALTITUDE
from environment_definition.constants.UNIT_REGISTRY import UREG as ureg
from environment_definition.mission_profiles.s01_multiple_targets_fwd_fish import (
    S01_SECONDARY_CAMERA,
    S01_SECONDARY_MOUNT,
)

scnd_camera = S01_SECONDARY_CAMERA
scnd_mount = S01_SECONDARY_MOUNT

ref_altitude = CAMERA_ALTITUDE
gsd = scnd_camera.gsd_at(ref_altitude)
swath_y = scnd_camera.swath_at(ref_altitude, axis="y")
swath_x = scnd_camera.swath_at(ref_altitude, axis="x")

print("Secondary camera (GoPro-style)")
print(f"  mount tilt off nadir: {scnd_mount.tilt_off_nadir.to('deg'):~}")
print(f"  focal length:         {scnd_camera.focal_length.to('mm'):~}")
print(f"  pixel pitch:          {scnd_camera.pixel_size.to('um'):~}")
print(f"  resolution:           {scnd_camera.n_pixels_x} x {scnd_camera.n_pixels_y}")
print(f"  FOV (y / x):          {scnd_camera.fov(axis='y').to('deg'):~} / {scnd_camera.fov(axis='x').to('deg'):~}")
print(f"  GSD @ {ref_altitude.to('km'):~}:           {gsd.to('m'):~}")
print(f"  swath (y / x):        {swath_y.to('km'):~} / {swath_x.to('km'):~}")


Secondary camera (GoPro-style)
  mount tilt off nadir: 25 deg
  focal length:         3.307162339212715 mm
  pixel pitch:          1.55 µm
  resolution:           5312 x 2988
  FOV (y / x):          70.0 deg / 102.44785777798826 deg
  GSD @ 550 km:           257.773859448019 m
  swath (y / x):        770.2282920306807 km / 1369.294741387877 km


## Camera mount verification

Use this section to explore camera mount tilt and field of view without changing simulation logic.
Change the `tilt_off_nadir`, `fov_y`, and pixel geometry below to add a third camera or compare different mount angles.

In [4]:
from simulation.camera_image import CameraImage, CameraMount, DEFAULT_NADIR_CAMERA

CAMERA_OPTIONS = [
    dict(
        name="Primary nadir camera",
        camera=DEFAULT_NADIR_CAMERA,
        mount=CameraMount(camera=DEFAULT_NADIR_CAMERA, tilt_off_nadir=0 * ureg.deg),
    ),
    dict(
        name="Secondary forward-looking camera",
        camera=scnd_camera,
        mount=scnd_mount,
    ),
]

TERT_FOV_ANGLE = 45 * ureg.deg
TERT_PIXEL_SIZE = 1.2 * ureg.um
TERT_N_PIXELS_X = 4096
TERT_N_PIXELS_Y = 2160
TERT_TILT_ANGLE = 15 * ureg.deg

tertiary_camera = CameraImage.from_fov(
    fov_y=TERT_FOV_ANGLE,
    pixel_size=TERT_PIXEL_SIZE,
    n_pixels_x=TERT_N_PIXELS_X,
    n_pixels_y=TERT_N_PIXELS_Y,
    axis="x",
    exposure_time=100 * ureg.millisecond,
)
tertiary_mount = CameraMount(camera=tertiary_camera, tilt_off_nadir=TERT_TILT_ANGLE)
CAMERA_OPTIONS.append(
    dict(
        name="Tertiary test camera",
        camera=tertiary_camera,
        mount=tertiary_mount,
    )
)


def describe_camera_mount(name, camera, mount, altitude):
    fov_y = camera.fov(axis="y").to(ureg.deg)
    fov_x = camera.fov(axis="x").to(ureg.deg)
    gsd = camera.gsd_at(altitude)
    swath_y = camera.swath_at(altitude, axis="y")
    swath_x = camera.swath_at(altitude, axis="x")

    print(f"{name}")
    print(f"  tilt off nadir: {mount.tilt_off_nadir.to('deg'):~}")
    print(f"  FOV (y / x):    {fov_y:~} / {fov_x:~}")
    print(f"  GSD @ {altitude.to('km'):~}: {gsd.to('m'):~}")
    print(f"  swath (y / x):  {swath_y.to('km'):~} / {swath_x.to('km'):~}")
    print()

for option in CAMERA_OPTIONS:
    describe_camera_mount(option["name"], option["camera"], option["mount"], ref_altitude)

Primary nadir camera
  tilt off nadir: 0 deg
  FOV (y / x):    1.2027913100169598 deg / 1.605508475144562 deg
  GSD @ 550 km: 1.6494845360824741 m
  swath (y / x):  11.54639175257732 km / 15.41278350515464 km

Secondary forward-looking camera
  tilt off nadir: 25 deg
  FOV (y / x):    70.0 deg / 102.44785777798826 deg
  GSD @ 550 km: 257.773859448019 m
  swath (y / x):  770.2282920306807 km / 1369.294741387877 km

Tertiary test camera
  tilt off nadir: 15 deg
  FOV (y / x):    24.643498296453373 deg / 45.00000000000001 deg
  GSD @ 550 km: 111.23899380136831 m
  swath (y / x):  240.27622661095555 km / 455.6349186104046 km



## Minimal coast camera verification

Run a short headless coast to verify the current camera setup is propagated through the simulation and produces 1D observation lines for primary and secondary cameras. This uses the existing `build_setup(..., include_cameras=True)` path and does not introduce new camera logic.

In [5]:
from environment_definition.constants.SIMULATION import RenderMode, SimulationConfig
from environment_definition.mission_profiles.s01_multiple_targets_fwd_fish import build_setup
from simulation.camera_2d import observation_codes_to_ascii_line
from simulation.run_simulation import run_simulation


def minimal_camera_coast_test(*, seed: int = 0, max_steps: int = 10, stride: int = 2) -> None:
    setup = build_setup(seed=seed, include_cameras=True)
    sim_cfg = SimulationConfig(render_mode=RenderMode.HEADLESS, builtin_torque_policy="coast")
    series = run_simulation(setup=setup, simulation_config=sim_cfg)

    n = min(series.t_s.shape[0], max_steps)
    print(f"Minimal coast verification: {n} steps, stride={stride}")
    print(f"primary bins={series.camera_observation_line_codes.shape[1]}")
    print(f"secondary bins={series.secondary_camera_observation_line_codes.shape[1]}")
    print(f"secondary tilt rad={series.secondary_camera_tilt_off_nadir_rad:.3f}")
    print(f"secondary vfov rad={series.secondary_camera_vertical_fov_rad:.3f}\n")

    if series.secondary_camera_observation_line_codes.size == 0:
        print("WARNING: no secondary camera observation line codes were produced.")

    for k in range(0, n, stride):
        print(f"step {k}: primary")
        print(observation_codes_to_ascii_line(series.camera_observation_line_codes[k]))
        if series.secondary_camera_observation_line_codes.shape[1] > 0:
            print(f"step {k}: secondary")
            print(observation_codes_to_ascii_line(series.secondary_camera_observation_line_codes[k]))
        print()


minimal_camera_coast_test(max_steps=6, stride=1)

d:\code\sem-proj-asc\backend\simulation\stepper.py:164: UserWarning: controller_update_interval (1 s) is not an integer multiple of simulation_timestep (0.4 s); using nearest multiple: 0.8 s (2 sim steps).
  self._controller_interval_steps, self._effective_controller_interval_s = resolve_controller_interval_steps(


╭──────────────────────────────────────────────── Simulation info ────────────────────────────────────────────────╮
│  Parameter                                     Value                                                            │
│  episode duration                              774.76 s                                                         │
│  simulation timestep                           0.399979 s                                                       │
│  integration steps                             1937 (+1 state samples)                                          │
│  orbit altitude                                548.22 km                                                        │
│  orbit period                                  5727.9 s                                                         │
│  theta center offset                           90.00 deg                                                        │
│  episode theta start (rel. center)             -23.364 deg                                                      │
│  episode theta end (rel. center)               23.011 deg                                                       │
│  sat motion span scale                         1.050                                                            │
│  sat z offset                                  0.00 deg                                                         │
│  target areas                                  1                                                                │
│  target phi stripe                             89.65 deg .. 90.00 deg                                           │
│  cloud patches                                 2                                                                │
│  render mode                                   headless                                                         │
│  torque command source                         builtin                                                          │
│  torque policy                                 coast                                                            │
│  attitude controller                           disabled                                                         │
│  control stack (display)                       builtin:coast                                                    │
│  controller seed                               -                                                                │
│  controller update (configured)                1 s                                                              │
│  controller update (effective)                 0.8 s (2 steps)                                                  │
│  reaction-wheel torque max                     0.1000 N*m                                                       │
│  attitude safety cutoff                        |omega_sat| > 3.00 deg/s -> block opposing torque                │
│  camera kernel backend                         accelerated                                                      │
│  reward shaping                                distance_reward, outer_gate                                      │
│  Camera 1 (primary) - role                     shapes reward (observation line + strip)                         │
│  Camera 1 (primary) - tilt off nadir           0.00 deg                                                         │
│  Camera 1 (primary) - FOV (cross x along)      1.61 deg x 1.20 deg                                              │
│  Camera 1 (primary) - resolution               9344 x 7000 px                                                   │
│  Camera 1 (primary) - pixel pitch              3.20 um                                                          │
│  Camera 1 (primary) - focal length             1067.0 mm                                                        │
│  Camera 1 (primary) - GSD @ 548.2 km           1.644 m                                                          │
│  Camera 1 (primary) - observation line bins    100    

Running simulation: 100%|██████████| 1937/1937 [00:08<00:00, 231.46step/s]

Minimal coast verification: 6 steps, stride=1
primary bins=100
secondary bins=200
secondary tilt rad=0.436
secondary vfov rad=1.222

step 0: primary
EEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEE
step 0: secondary
EEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEE

step 1: primary
EEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEE
step 1: secondary
EEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEE

step 2: primary
EEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEE
step 2: secondary
EEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEE

## Render and export simulation video

Render the headless coast episode to an MP4 so the secondary camera placement can be reviewed visually. The video is saved to the notebook folder and displayed inline if the notebook environment supports HTML5 video.

In [6]:
import matplotlib

import camera_verification
from utils.notebook.video import export_and_play_saved_video

setup = build_setup(seed=0, include_cameras=True)
sim_cfg = SimulationConfig(render_mode=RenderMode.HEADLESS, builtin_torque_policy="coast")
series = run_simulation(setup=setup, simulation_config=sim_cfg)
video_path = ARTIFACT_DIR / "04-camera-coast.mp4"
print(f"Exporting camera verification video to {video_path}")
export_and_play_saved_video(simulation_series=series, out_path=video_path)
camera_verification.print_dual_camera_numeric_gate(series)
matplotlib.use("module://matplotlib_inline.backend_inline")
camera_verification.show_dual_camera_mount_panels(series, frame_idx=0)
print(f"artifact={video_path}")


ModuleNotFoundError: No module named 'camera_verification'

## Visual verification gate

Human review (after re-export once `.gitignore` allows committed artifacts):

- [ ] **Artifact:** `backend/notebooks/s01/artifacts/04-camera-coast.mp4`
- [ ] **Primary cone:** narrow nadir FOV (filled), body +Z along orbit normal at start
- [ ] **Secondary cone:** gold dashed, tilted ~25° prograde (forward-looking)
- [ ] **1D strips:** primary (100 bins) and secondary (200 bins) panels update in the video
- [ ] **Numeric gate:** stdout shows secondary tilt 25° and vertical FOV 70°